In [ ]:
import os
import sys
import re
from typing import Union
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

PATH = os.path.dirname(os.path.abspath(""))
DATA_PATH = os.path.join(os.path.dirname(PATH), "data")

sys.path.append(PATH)

import src.dashboard_helpers.dashboard_utils as ut
import src.dashboard_plots.dashboard_plots_comparison as dbp
import src.dashboard_plots.dashboard_plots_summary as dbps

In [ ]:
data = pd.read_csv(DATA_PATH + '/full_data_2026-01-09.csv')
print(data.shape)
data.head()

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp['played_at'] = pd.to_datetime(tmp['played_at']).dt.strftime('%Y-%m-%d %H:%M:%S')
tmp

In [ ]:
pd.to_datetime(tmp['played_at'][0])

## Data Cleaning

- simplify artist_genres column
    - only use first 
    - only use most prevalent
- fill missing values
    - if a song (same song title + same combined_artist_names) occures more than once but one (or more) are lacking audio features
- convert played_at to Datetime 

#### Fill missing feature values

If a song (same song title + same combined_artist_names) occurs more than once but one (or more) are lacking audio features, then fill the missing values accordingly.

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp

In [ ]:
# Convert played_at to datetime
data['played_at'] = pd.to_datetime(data['played_at'],format='ISO8601').dt.strftime('%Y-%m-%d %H:%M:%S')
data = ut.fill_missing_features(data=data)
data = ut.assign_main_genre(data=data)
print(data.shape)
data = ut.assign_simplified_main_genre(data=data)
print(data.shape)

In [ ]:
tmp = data.loc[data['first_artist_name']=='Killswitch Engage'].reset_index(drop=True)
tmp

## Data Enrichment

- track_duration_ms to track_duration_sec
- track_duration_ms to track_duration_min_sec (mm:ss)
- seconds_till_next_song
    - sort all data points by played_at (in ascending order)
    - calculate delta between current and next song in seconds
- song_fully_played_flag
    - IF track_duration_sec 0.95 <= seconds_till_next_song THEN 1 ELSE 0
        - we multiply with 0.95 here account for potential inaccuracies of played_at timestamp
        - besides, if the user listened to 90% of the song, we can consider it fully played

In [ ]:
data = ut.convert_track_duration(data=data)
data = ut.add_fully_played_label(data=data, fully_played_threshold=0.95)
data

## Data Analysis

### 1. Summary Stats:

In [ ]:
dbps.create_summary_stats(data=data, cutoff_date='2025-01-09')

### 2. Top 5's

In [ ]:
# Top 5 Songs (incl. album image-url)
top5_tracks_df = dbps.create_top5_input_data(data=data, cutoff_date='2025-01-09', tracks=True, sort_by_track_time=True)
top5_tracks_df

In [ ]:
# Top 5 Artists (incl. artist image-url)
top5_artists_df = dbps.create_top5_input_data(data=data, cutoff_date='2025-01-09', tracks=False, sort_by_track_time=True)
top5_artists_df

### 3. Line Chart Inputs

In [ ]:
cutoff_date = '2025-11-09'

agg_df_dict = dbps._create_input_data_summary_sheet_plots(data=data, cutoff_date=cutoff_date)
print(agg_df_dict.keys())

In [ ]:
agg_df_dict['df_agg_daily_genre']

### 4. Bar Charts

#### 4.1. Overall Bar Chart

In [ ]:
agg_df_dict.keys()

In [ ]:
cutoff_date = '2025-11-09'
kpi = 'track_time_played_mins' #'num_songs_played' #'track_time_played_mins'
agg_level_daily = True

dbps.listening_summary_bar_chart_overall(
    data=data, 
    cutoff_date=cutoff_date, 
    agg_level_daily=agg_level_daily,
    kpi=kpi
)

#### 4.2. Stacked Bar Chart

In [ ]:
dbps.listening_summary_stacked_bar_chart_genre(
    data=data, cutoff_date=cutoff_date, agg_level_daily=True, kpi='track_time_played_mins'
)

#### 4.3. Top 10 Genres (overall)

In [ ]:
dbps.listening_summary_genre_chart_overall(
    data=data, cutoff_date=cutoff_date, kpi='track_time_played_mins'
)